In [1]:
import pandas as pd
import numpy as np

In [ ]:
import prodec

In [ ]:
descr = prodec.ProteinDescriptors()

In [6]:
targets = pd.read_csv('data/saifudeen_2026_raw/kinase_uniprot_target_mapping.csv')

In [7]:
sequences = targets['Entry']

In [ ]:
import io
import requests
from Bio import SeqIO
from Bio.Align import MultipleSeqAlignment
# Biopython's modern alignment interface
from Bio.Align.Applications import MuscleCommandline  

# 1. Define your UniProt Accession IDs
accessions = list(sequences) # Example: p53, HRAS, Calmodulin

print(f"Fetching {len(accessions)} sequences from UniProt...")
records = []

# 2. Fetch FASTA strings directly via UniProt's REST API
for acc in accessions:
    url = f"https://www.uniprot.org/uniprotkb/{acc}.fasta"
    
    response = requests.get(url)
    print(acc)
    if response.status_code == 200:
        # Parse the plain text FASTA into a Biopython SeqRecord
        try:
            fasta_io = io.StringIO(response.text)
            record = SeqIO.read(fasta_io, "fasta")
            records.append(record)
        except:
            print('failed for unknown reason')
            records.append(np.nan)
    else:
        print(f"Failed to fetch {acc}. Status code: {response.status_code}")

# 3. Save sequences to a local file for the alignment tool
unaligned_fasta = "unaligned_sequences.fasta"
SeqIO.write(records, unaligned_fasta, "fasta")
print(f"Saved unaligned sequences to '{unaligned_fasta}'")


In [ ]:
import pandas as pd
from prodec import ProteinDescriptors

# 1. Load your aligned data
df = pd.read_csv("aligned_kinases.csv")

# 2. Instantiate ProDEC and select your descriptor (e.g., Z-scales)
pdescs = ProteinDescriptors()
zscales = pdescs.get_descriptor('Zscale Hellberg')

print(f"Generating features for {len(df)} sequences...")

# 3. Apply prodec across the column
# We use gaps='omit' if you don't want alignment hyphens counted as 0.0
descriptor_lists = df['Aligned_Sequence'].apply(lambda seq: zscales.get(seq, gaps='omit'))

# 4. Neatly expand the lists into individual feature columns
features_df = pd.DataFrame(list(descriptor_lists))

# 5. Prefix the column names so you know what they are (e.g., feature_0, feature_1...)
features_df.columns = [f"zscale_{i}" for i in range(features_df.shape[1])]

# 6. Stitch it back to your original IDs
final_df = pd.concat([df['Sequence_ID'], features_df], axis=1)

# Save your neat ML-ready matrix
final_df.to_csv("kinase_features_matrix.csv", index=False)
print("✅ Done! Saved matrix to 'kinase_features_matrix.csv'")

In [ ]:
# Split the 'Sequence_ID' column at the first underscore
final_df[['entry', 'sequence_section']] = final_df['Sequence_ID'].str.split('_', n=1, expand=True)

# Reorder the columns neatly so your identifiers are at the front
cols = ['entry', 'sequence_section'] + [c for c in final_df.columns if c not in ['Sequence_ID', 'entry', 'sequence_section']]
final_df = final_df[cols]

# Save the final structured file
final_df.to_csv("kinase_features_matrix.csv", index=False)

In [80]:
import pandas as pd

In [111]:
protein_descriptors = pd.read_csv('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.csv')

In [113]:

protein_descriptors['protein_descriptor'] = protein_descriptors['protein_descriptor'].apply(ast.literal_eval) 
protein_descriptors['accession'] = protein_descriptors['Entry']
protein_descriptors[['protein_descriptor', 'accession']].to_pickle('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.pkl')

In [86]:
test = pd.read_pickle('/home/boefma/auxiliary_ranking/data/protein_data/CMF_Zscales.pkl')

In [91]:
retest = pd.read_pickle('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.pkl')

In [98]:
print(test)
print(retest)

                                    protein_descriptor  target_id
0    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 2.29, ...  Q96PF2_WT
1    [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 2.39,...  P15056_WT
2    [-2.59, -2.64, -1.54, 2.05, -4.06, 0.36, 2.39,...  Q16539_WT
3    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 3.11, ...  P49760_WT
4    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 3.11, ...  P22455_WT
..                                                 ...        ...
352  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q00532_WT
353  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 1.75,...  Q99986_WT
354  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q8IZL9_WT
355  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q96PN8_WT
356  [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 2.39, ...  Q6ZWH5_WT

[357 rows x 2 columns]
                                    protein_descriptor   Entry
0    [-0.021136, -0.246364, -0.031818, -0.086742, -...  Q2M2I8
1    [-0.142955, -0.105227, -0.095568, 0.012022, -0...  Q6

In [97]:
print(retest.dtypes)

protein_descriptor    object
Entry                 object
dtype: object


In [107]:
import ast
from sklearn.preprocessing import MinMaxScaler, StandardScaler
scaler = StandardScaler()
retest['protein_descriptor'] = retest['protein_descriptor'].apply(ast.literal_eval) 
scaler = scaler.fit(np.array(retest['protein_descriptor'].to_list()))
print(retest)

                                    protein_descriptor   Entry
0    [-0.021136, -0.246364, -0.031818, -0.086742, -...  Q2M2I8
1    [-0.142955, -0.105227, -0.095568, 0.012022, -0...  Q6ZMQ8
2    [0.086364, -0.217045, -0.101364, 0.035056, -0....  P00519
3    [0.086364, -0.217045, -0.101364, -0.029775, -0...  P42684
4    [0.0025, -0.213295, -0.082841, 0.026629, -0.04...  Q04771
..                                                 ...     ...
478  [-0.010909, -0.075341, -0.094886, 0.017191, -0...  Q9Y3S1
479  [-0.032386, -0.076477, -0.100341, 0.017191, -0...  Q9BYP7
480  [-0.01375, -0.082727, -0.095455, 0.08764, -0.1...  Q96J92
481  [-0.020455, -0.187727, -0.074318, -0.023371, -...  P07947
482  [-0.054205, -0.226477, 0.053182, 0.096517, -0....  P43403

[483 rows x 2 columns]
